# SonarVision Physics — Interactive Notebook

Deterministic underwater sonar physics using the FLUX 9-opcode engine.

```bash
pip install sonar-vision-physics
```

In [ ]:
from sonar_vision_physics import compute_physics, dive_profile, FluxPhysics
import matplotlib.pyplot as plt
import numpy as np

print("SonarVision Physics v1.0.1 — all imports OK")

## 1. Single Depth Ping

Query all 9 FLUX physics parameters at a given depth.

In [ ]:
ping = compute_physics(depth=25.0, chlorophyll=3.0, season='summer')
for k, v in ping.items():
    print(f"  {k:>20}: {v}")

## 2. Full Dive Profile

Compute a complete 0-100m descent at 1m resolution.

In [ ]:
profile = dive_profile(0, 100, 2, chl=8.0, season='summer')
depths = [p['depth'] for p in profile]
temps = [p['temperature'] for p in profile]
vis = [p['visibility'] for p in profile]
ss = [p['sound_speed'] for p in profile]

print(f"Profile: {len(profile)} depths (0-100m, step=2m)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('FLUX Dive Profile — Summer, Chlorophyll=8.0', fontsize=14)

axes[0].plot(temps, depths, 'b-', linewidth=2)
axes[0].set_ylabel('Depth (m)')
axes[0].set_xlabel('Temperature (°C)')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

axes[1].plot(vis, depths, 'c-', linewidth=2)
axes[1].set_xlabel('Visibility (m)')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

axes[2].plot(ss, depths, 'g-', linewidth=2)
axes[2].set_xlabel('Sound Speed (m/s)')
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Seasonal Comparison

Compare summer vs winter profiles. Note the thermocline shift.

In [ ]:
summer = dive_profile(0, 100, 2, chl=5.0, season='summer')
winter = dive_profile(0, 100, 2, chl=5.0, season='winter')

plt.figure(figsize=(8, 6))
plt.plot([p['temperature'] for p in summer], [p['depth'] for p in summer], 'r-', linewidth=2, label='Summer')
plt.plot([p['temperature'] for p in winter], [p['depth'] for p in winter], 'b-', linewidth=2, label='Winter')
plt.gca().invert_yaxis()
plt.xlabel('Temperature (°C)')
plt.ylabel('Depth (m)')
plt.title('Seasonal Thermocline Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Key difference at 15m
s15 = summer[8]  # index 8 = 16m
w15 = winter[8]
print(f"Summer @16m: {s15['temperature']:.1f}°C, Sound: {s15['sound_speed']:.0f} m/s")
print(f"Winter @16m:  {w15['temperature']:.1f}°C, Sound: {w15['sound_speed']:.0f} m/s")

## 4. Sediment Comparison

Compare seabed reflectivity across different sediment types.

In [ ]:
physics = FluxPhysics()
sediments = ['mud', 'sand', 'gravel', 'rock', 'seagrass']
sed_map = {'mud': 0, 'sand': 1, 'gravel': 2, 'rock': 3, 'seagrass': 4}

for sed in sediments:
    rs = [physics.compute(d, sediment=sed_map[sed])['seabed_reflectivity'] for d in range(0, 101, 5)]
    plt.plot(range(0, 101, 5), rs, label=sed, linewidth=2)

plt.xlabel('Depth (m)')
plt.ylabel('Reflectivity')
plt.title('Seabed Reflectivity by Sediment Type')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. Visibility vs Absorption

Attenuation limits visibility. The Secchi depth emerges naturally.

In [ ]:
profile = dive_profile(0, 100, 1, chl=12.0, season='summer')
dep = [p['depth'] for p in profile]
attens = [p['attenuation'] for p in profile]
viss = [p['visibility'] for p in profile]

fig, ax1 = plt.subplots(figsize=(8, 6))

color = 'tab:red'
ax1.set_xlabel('Depth (m)')
ax1.set_ylabel('Attenuation (m⁻¹)', color=color)
ax1.plot(dep, attens, color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:cyan'
ax2.set_ylabel('Visibility (m)', color=color)
ax2.plot(dep, viss, color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Coastal Water (Chlorophyll=12): Attenuation vs Visibility')
fig.tight_layout()
plt.show()

print(f"Surface visibility: {viss[0]:.1f}m")
print(f"Max visibility:     {max(viss):.1f}m")
print(f"Mean attenuation:   {np.mean(attens):.3f} m⁻¹")

## 6. WebSocket Streaming Demo

Connect to the streaming server from notebook:

In [ ]:
# Start a temporary server and connect
from sonar_vision_physics.streaming import StreamingServer, DiveSimulator
import asyncio
import json

from threading import Thread

frames_collected = []

async def collect_samples():
    sim = DiveSimulator(max_depth=50.0, rate_hz=10.0)
    for _ in range(5):
        frames_collected.append(sim.step())
        await asyncio.sleep(0.1)

asyncio.run(collect_samples())

print(f"Collected {len(frames_collected)} frames:")
for f in frames_collected:
    print(f"  Frame {f['frame_id']}: depth={f['depth']:.1f}m  temp={f['temperature']:.1f}C")

## 7. Determinism Verification

FLUX physics is deterministic: same inputs → same outputs.

In [ ]:
physics = FluxPhysics()
hash1 = hash(str(physics.compute(37.0, chl=2.5, sediment=0)))
hash2 = hash(str(physics.compute(37.0, chl=2.5, sediment=0)))
hash3 = hash(str(physics.compute(37.0, chl=2.5, sediment=0)))

print(f"Run 1 hash: {hash1}")
print(f"Run 2 hash: {hash2}")
print(f"Run 3 hash: {hash3}")
print(f"Deterministic: {hash1 == hash2 == hash3}")

## Summary

- **FLUX 9-opcode physics engine** — deterministic, bit-identical
- **Seasonal thermocline** — summer peak at 15m (22°C), winter at 40m (8°C)
- **Sediment reflectivity** — rock > gravel > sand > mud > seagrass
- **Attenuation drives visibility** — Secchi depth emerges from physics
- **Cross-platform** — Python (PyPI), TypeScript (npm), Rust (crates.io), C (MEP bridge)